# Replicating prefix-tuning with the authors' original code (E2E)
COMP8240 final project, Niharika Kunaminna. Code: https://github.com/XiangLi1999/PrefixTuning (branch `cleaned`).

**Runtime → Change runtime type → T4 GPU** before running. If Colab restarts, run cells 1–3 again (everything lives in `/content`).

Problems found and fixed (for the report):
1. Colab's Python 3.13 cannot build `tokenizers==0.8.1rc2`, which the bundled `transformers==3.2.0` needs → a Python 3.8 environment made with `uv` (plus `six`, which the old library imports but does not declare).
2. `train_e2e.py` starts `python run_language_modeling.py` with `os.system`, which picked up Colab's Python 3.13 and new transformers (`ImportError: cannot import name 'cached_path'`) → call the 3.8 Python directly.
3. Hard-coded paths on the authors' servers (`/u/scr/xlisali/...`) for the model cache, test data, output folder and the E2E scorer → GPT-2 medium downloaded to `/content/gpt2-medium`, paths patched.
4. `gen.py` reads its settings from the model folder's name (it needs `prefixtune` in it, and it hard-codes prefix length 20) → `NameError: name 'tuning_mode' is not defined` → call `run_generation.py` directly with the right settings.
5. `run_generation.py` also reads the prefix format (`cat`, `peek`, ...) from the folder name and stops with an `AssertionError` otherwise → link the model folder as `e2e_p10_e1_act_cat`.

In [ ]:
%%bash
# 1. Environment (skips steps already done)
set -e
nvidia-smi --query-gpu=name,memory.total --format=csv
cd /content
[ -d PrefixTuning ] || git clone -q -b cleaned https://github.com/XiangLi1999/PrefixTuning
if [ ! -x /content/py38/bin/python ]; then
  pip -q install uv
  uv venv /content/py38 --python 3.8
  uv pip install --python /content/py38/bin/python torch==1.13.1 nltk sentencepiece -e /content/PrefixTuning/transformers
fi
# the bundled transformers 3.2.0 also imports six, which is not in its install list
uv pip install --python /content/py38/bin/python six sacremoses
/content/py38/bin/python -c "import torch, transformers; print('torch', torch.__version__, 'cuda', torch.cuda.is_available(), 'transformers', transformers.__version__)"
python --version

In [ ]:
# 2. GPT-2 medium weights (the code expects them in a cache folder on the authors' server)
import os
from huggingface_hub import snapshot_download
if not os.path.exists("/content/gpt2-medium/pytorch_model.bin"):
    snapshot_download("gpt2-medium", local_dir="/content/gpt2-medium",
                      allow_patterns=["config.json", "vocab.json", "merges.txt", "pytorch_model.bin"])
print(os.listdir("/content/gpt2-medium"))

In [ ]:
%%bash
# 3. Patch the hard-coded paths in run_generation.py (original kept as run_generation_orig.py)
cd /content/PrefixTuning/gpt2
[ -f run_generation_orig.py ] || cp run_generation.py run_generation_orig.py
sed -e 's#/u/scr/xlisali/e2e_data/#/content/PrefixTuning/data/e2e_data/#g' \
    -e 's#/u/scr/xlisali/contrast_LM/transformers/examples/text-generation/#/content/gen_out/#g' \
    -e 's#python /u/scr/xlisali/e2e-metrics/measure_scores.py#/usr/bin/python3 /content/e2e-metrics/measure_scores.py#g' \
    run_generation_orig.py > run_generation.py
mkdir -p /content/gen_out/e2e_results
echo "changed lines:"; diff run_generation_orig.py run_generation.py | grep -c '^>'
ls /content/PrefixTuning/data/e2e_data/

## 4. (Recommended) a longer training run: 1 epoch, prefix length 10
Your first run trained for only 300 steps (1,500 examples, 3.6% of one epoch) with prefix length 5. 300 steps took about 1 minute, so one full epoch (8,413 steps) should take **about 30 minutes** on a T4. Prefix length 10 is the paper's table-to-text setting. Evaluation is switched off (it took 13 minutes last time and is not needed). Skip this cell if you are short of time and go to cell 5 with `short_run`.

The folder name deliberately does **not** contain `prefixtune`: after training, `run_language_modeling.py` automatically calls `gen.py`, which would otherwise launch a generation run with the wrong prefix length (20). With this name `gen.py` stops immediately with the same `NameError` as before, which is harmless.

In [ ]:
%%bash
cd /content/PrefixTuning/gpt2
/content/py38/bin/python run_language_modeling.py \
  --output_dir=save_e2e_models_convcheck/e2e_p10_e1 --model_type=gpt2 \
  --model_name_or_path=/content/gpt2-medium --tokenizer_name=/content/gpt2-medium \
  --per_device_train_batch_size 5 --per_device_eval_batch_size 5 --save_steps 500000 --num_train_epochs 1 \
  --do_train --train_data_file=../data/e2e_data/src1_train.txt --line_by_line --save_total_limit 1 --overwrite_output_dir \
  --task_mode data2text --eval_data_file=../data/e2e_data/src1_valid.txt --tuning_mode prefixtune \
  --logging_dir save_e2e_models_convcheck/runs/e2e_p10_e1 --train_embs no --optim_prefix yes --preseqlen 10 \
  --prefix_mode activation --format_mode cat --gradient_accumulation_steps 1 --learning_rate 5e-05 --weight_decay 0.0 \
  --seed 101 --disable_tqdm --mid_dim 512 --init_random no --use_dropout no --prefix_dropout 0.0 --objective_mode 1 \
  --logging_steps 500 --cache_dir /content/cache > /content/train_p10_e1.log 2>&1
grep -E "loss|Num examples|optimization steps|total param|Training completed" /content/train_p10_e1.log | tail -25

## 5. Generate for the 630 E2E test MRs (beam search, beam 5)
Takes roughly 15 minutes.

In [ ]:
%%bash
cd /content/PrefixTuning/gpt2
# run_generation.py also reads the prefix format from the folder name (it must contain 'cat'): link the folder under such a name
(cd save_e2e_models_convcheck && ln -sfn e2e_p10_e1 e2e_p10_e1_act_cat)
MODEL=save_e2e_models_convcheck/e2e_p10_e1_act_cat
PRE=10
/content/py38/bin/python run_generation.py --model_type=gpt2 --length 100 \
  --model_name_or_path=/content/gpt2-medium --tokenizer_name=$MODEL \
  --num_return_sequences 5 --stop_token [EOS] --task_mode=data2text --control_mode=yes \
  --tuning_mode prefixtune --gen_dir e2e_results --eval_dataset test \
  --optim_prefix yes --preseqlen $PRE --prefix_mode activation --format_mode cat \
  --prefixModel_name_or_path $MODEL --cache_dir /content/cache > /content/gen_$(basename $MODEL).log 2>&1
tail -5 /content/gen_$(basename $MODEL).log
ls -la /content/gen_out/e2e_results/
wc -l /content/gen_out/e2e_results/*_test_beam

## 6. Score with the same scorer as my peft runs (directly comparable with Table 1 of the report)

In [ ]:
%%writefile /content/score_authors_output.py
"""
Score the E2E output of the authors' code (run_generation.py) with the SAME scorer as src/train_prefix.py,
so the authors'-code run and my peft run can be compared directly.

The authors' code writes two files:
  <name>_test_gold  : the human references, one per line, a blank line after each MR's group
  <name>_test_beam  : one generated sentence per MR, in the same order

Usage:
  python src/score_authors_output.py /content/gen_out/e2e_results/short_run_test_gold \
                                     /content/gen_out/e2e_results/short_run_test_beam
"""
import json
import re
import sys


def read_gold(path):
    groups, cur = [], []
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                cur.append(line)
            elif cur:
                groups.append(cur)
                cur = []
    if cur:
        groups.append(cur)
    return groups


def tok(s):
    return re.findall(r"\w+|[^\w\s]", s.lower())


def main(gold_path, out_path):
    refs_list = read_gold(gold_path)
    with open(out_path, encoding="utf-8") as f:
        hyps = [l.strip() for l in f]
    hyps = [h for h in hyps if h != ""][: len(refs_list)]
    assert len(hyps) == len(refs_list), f"{len(hyps)} outputs vs {len(refs_list)} reference groups"

    from nltk.translate.bleu_score import corpus_bleu
    from nltk.translate.nist_score import corpus_nist
    from rouge_score import rouge_scorer
    import nltk
    nltk.download("wordnet", quiet=True)
    from nltk.translate.meteor_score import meteor_score

    H = [tok(h) for h in hyps]
    R = [[tok(r) for r in refs] for refs in refs_list]
    sc = rouge_scorer.RougeScorer(["rougeL"])
    res = {
        "n_MRs": len(hyps),
        "BLEU": 100 * corpus_bleu(R, H),
        "NIST": corpus_nist(R, H, n=5),
        "METEOR": 100 * sum(meteor_score(rt, ht) for ht, rt in zip(H, R)) / len(H),
        "ROUGE_L": 100 * sum(max(sc.score(r, h)["rougeL"].fmeasure for r in refs)
                             for h, refs in zip(hyps, refs_list)) / len(hyps),
        "n_Error_outputs": sum(h == "Error" for h in hyps),
    }
    print(json.dumps(res, indent=2))


if __name__ == "__main__":
    main(sys.argv[1], sys.argv[2])


In [ ]:
%%bash
pip -q install nltk rouge-score
cd /content/gen_out/e2e_results
for OUT in *_test_beam; do
  NAME=${OUT%_test_beam}
  echo "== $NAME"
  python /content/score_authors_output.py ${NAME}_test_gold $OUT | tee /content/${NAME}_scores.json
done
head -5 *_test_beam

## 7. (Optional) the official E2E scorer, as used in the paper
Needs Java for METEOR. If it fails to install on Colab's Python, record the error for the report and use the cell-6 numbers.

In [ ]:
%%bash
cd /content
[ -d e2e-metrics ] || git clone -q https://github.com/tuetschek/e2e-metrics
apt-get -qq install -y default-jre-headless > /dev/null 2>&1 || echo "java install failed"
pip -q install -r e2e-metrics/requirements.txt 2>&1 | tail -3
cd /content/gen_out/e2e_results
for OUT in *_test_beam; do
  NAME=${OUT%_test_beam}
  echo "== $NAME"
  /usr/bin/python3 /content/e2e-metrics/measure_scores.py ${NAME}_test_gold $OUT 2>&1 | tail -8
done

## 8. Save the outputs (download this zip and add it to your GitHub repo under `results/authors_code/`)

In [ ]:
%%bash
cd /content
mkdir -p authors_code_results
cp gen_out/e2e_results/* authors_code_results/ 2>/dev/null
cp *_scores.json train_p10_e1.log gen_*.log authors_code_results/ 2>/dev/null
zip -qr authors_code_results.zip authors_code_results
ls -la authors_code_results

In [ ]:
from google.colab import files
files.download('/content/authors_code_results.zip')